In [ ]:
# Linalg
import jax.numpy as np
import numpy as onp

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Data handling
import h5py as hf

# Analysis
from atomcorr.analysis.onsager_coefficients import OnsagerCoefficients

In [ ]:
main_prefix = "/tikhome/stovey/Downloads/lymburn-data"

In [ ]:
def normalize_covariance_matrix(covariance_matrix: np.ndarray):
    """
    Method for normalizing a covariance matrix.
    Returns
    -------
    normalized_covariance_matrix : np.ndarray
            A normalized covariance matrix, i.e, the matrix given if all of its inputs
            had been normalized.
    """
    order = np.shape(covariance_matrix)[0]

    diagonals = np.diagonal(covariance_matrix)

    repeated_diagonals = np.repeat(diagonals[None, :], order, axis=0)

    normalizing_matrix = np.sqrt(repeated_diagonals * repeated_diagonals.T)

    return covariance_matrix / normalizing_matrix

In [ ]:
params = np.linspace(0, 9, 10, dtype=int)
calculator = OnsagerCoefficients()
entropies = []
traces = []

for param in params:
    prefix = f"{main_prefix}/parameter-combination-{param}/seed-0"

    with hf.File(f"{prefix}/simulation_output_train.h5", "r") as db:
        data = db["agent_observables"]["agent_velocity"][:]

        full_correlation = calculator._compute_correlation_matrix(
            data, 
            data, 
            correlation_time=1,
            data_range=500
        )
        eigs = np.linalg.eigh(full_correlation)[0]
        eigs += 1.01 * abs(eigs.min())
        eigs /= eigs.sum()

        entropy = (-eigs * np.log(eigs)).sum()
        entropies.append(entropy)
        traces.append(np.trace(full_correlation))


In [ ]:
plt.plot(traces)

In [ ]:
entropies